# 02 — RL for Engineering Problems

**Module 4 · Week 8 · Phạm Ngọc Khánh**

This notebook covers:
- Designing state / action / reward for an engineering toy problem
- Using Stable-Baselines3 (PPO) for continuous or discrete control
- Analyzing the reward curve and failure cases
- Connecting RL to scientific/engineering research

---

In [ ]:
import numpy as np
import torch
import gymnasium as gym
from gymnasium import spaces
import matplotlib.pyplot as plt
from stable_baselines3 import PPO
from stable_baselines3.common.env_checker import check_env
from stable_baselines3.common.callbacks import EvalCallback

print(f'Gymnasium: {gym.__version__}')

## 1. Custom Engineering Environment

We implement a **toy thermostat control** problem:
- Agent controls heater power to reach and maintain a target temperature
- State: `[current_temp, target_temp, time_step]`
- Action: heater power level `∈ {0, 1, 2}` (low / medium / high)
- Reward: `-|current_temp - target_temp|`

In [ ]:
class ThermostatEnv(gym.Env):
    """Toy thermostat control environment."""

    def __init__(self, max_steps: int = 100):
        super().__init__()
        self.max_steps = max_steps

        # State: [current_temp (normalized), target_temp (normalized), step/max_steps]
        self.observation_space = spaces.Box(
            low=np.array([0.0, 0.0, 0.0], dtype=np.float32),
            high=np.array([1.0, 1.0, 1.0], dtype=np.float32)
        )

        # Action: 0=low (0W), 1=medium (50W), 2=high (100W)
        self.action_space = spaces.Discrete(3)

        self.temp_min = 15.0   # °C
        self.temp_max = 35.0   # °C
        self.heat_levels = [0.0, 50.0, 100.0]  # Watts
        self.cooling_rate = 2.0  # W/step natural cooling toward ambient (20°C)
        self.ambient = 20.0

    def _normalize_temp(self, T):
        return (T - self.temp_min) / (self.temp_max - self.temp_min)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.current_temp = self.np_random.uniform(self.temp_min, self.temp_max)
        self.target_temp = self.np_random.uniform(self.temp_min + 5, self.temp_max - 5)
        self.step_count = 0
        return self._get_obs(), {}

    def _get_obs(self):
        return np.array([
            self._normalize_temp(self.current_temp),
            self._normalize_temp(self.target_temp),
            self.step_count / self.max_steps
        ], dtype=np.float32)

    def step(self, action):
        heat = self.heat_levels[action]
        # Simple heat transfer model
        delta_T = (heat - self.cooling_rate * (self.current_temp - self.ambient)) * 0.01
        self.current_temp = np.clip(self.current_temp + delta_T, self.temp_min, self.temp_max)
        self.step_count += 1

        error = abs(self.current_temp - self.target_temp)
        reward = -error / 20.0  # normalize reward to [-1, 0]
        terminated = False
        truncated = self.step_count >= self.max_steps

        return self._get_obs(), reward, terminated, truncated, {}

# Verify the environment
env = ThermostatEnv()
check_env(env)
print('Environment check passed!')

obs, _ = env.reset()
print(f'Observation: {obs} (current_temp_norm, target_temp_norm, step_frac)')

## 2. Train PPO Agent

In [ ]:
# Create separate eval environment
eval_env = ThermostatEnv()
eval_callback = EvalCallback(
    eval_env,
    eval_freq=2000,
    best_model_save_path='./logs/thermostat/',
    log_path='./logs/thermostat/',
    verbose=0
)

model = PPO(
    'MlpPolicy',
    ThermostatEnv(),
    verbose=0,
    learning_rate=3e-4,
    n_steps=512,
    batch_size=64,
)

print('Training PPO on ThermostatEnv...')
model.learn(total_timesteps=50_000, callback=eval_callback, progress_bar=False)
print('Training complete!')

## 3. Plot Evaluation Results

In [ ]:
# --- TODO: Load and plot the evaluation log ---
# The eval log is saved at ./logs/thermostat/evaluations.npz
# Load with np.load and plot mean reward vs timestep


## 4. Visualize Trained Policy

In [ ]:
# Load best model
best_model = PPO.load('./logs/thermostat/best_model')

# Run one episode and visualize
env = ThermostatEnv(max_steps=100)
obs, _ = env.reset(seed=0)
target = env.target_temp

temps = [env.current_temp]
actions = []
rewards = []

for _ in range(100):
    action, _ = best_model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, _ = env.step(action)
    temps.append(env.current_temp)
    actions.append(action)
    rewards.append(reward)
    if terminated or truncated:
        break

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
ax1.plot(temps, label='Temperature (°C)')
ax1.axhline(target, color='red', linestyle='--', label=f'Target: {target:.1f}°C')
ax1.set_ylabel('Temperature (°C)')
ax1.legend()
ax1.grid(True)

ax2.step(range(len(actions)), actions, label='Heater action (0=low, 1=med, 2=high)')
ax2.set_ylabel('Action')
ax2.set_xlabel('Step')
ax2.set_yticks([0, 1, 2])
ax2.legend()
ax2.grid(True)

plt.title(f'Trained PPO policy — Total reward: {sum(rewards):.2f}')
plt.tight_layout()
plt.show()

## 5. Your Turn: Custom Engineering Problem

*(Complete as part of Week 8 assignment)*

Choose a different toy engineering problem and fill in the formulation:

| Component | Definition |
|-----------|------------|
| Problem | |
| State `S` | |
| Action `A` | |
| Reward `R` | |
| Termination | |

Then implement the environment as a `gym.Env` subclass below.

In [ ]:
# --- TODO: Implement your custom environment ---


## 6. Lab Research Connection

*(Complete as part of Week 8 assignment)*

**Identify a real problem in your lab that could be formulated as RL:**

State:

Action:

Reward:

Main challenge:

Would you actually use RL or a simpler approach? Why?